# Zbor · Lav: Aromanian and Romani word helpers for Romanian readers

**Veritas AI International High School Hackathon 2026 · Track: AI + Languages**
**Team:** Lavinia Ichim and Bianca-Mihaela Cajvan (Colegiul Național „Petru Rareș” Suceava), Alexandru-Eduard Bejenaru (Ovidius High School)

**Pitch:** Romanians think they can read Aromanian because it looks familiar, but many words mean something else. Even the name is a trap: in Aromanian *zbor* means **"word"**, while in Romanian it means **"flight"**. Zbor shows a Romanian reader what each Aromanian word really means, how sure it is, and which words are traps.

**Who it's for:** a Romanian speaker who wants to start learning Aromanian, for example a teenager whose grandparents speak it. Because Aromanian looks so familiar, they keep guessing words wrong. Zbor shows them, word by word, what a text really says and which words to be careful with.

**What goes in → what comes out:** Aromanian text → every word with its most likely Romanian meaning, coloured by confidence (green = sure, yellow = unsure, red = unknown or guessed), with ⚠ on look-alike traps.

**How to run:** Runtime → Run all. No GPU needed; it takes about 3–5 minutes. The main corpus is gated, so either upload `aromanian-romanian-MT-corpus-limited.csv` (from the dataset page) to the Colab files panel, or paste a Hugging Face read token when asked (section 1).

**Main results (test sets, used only once):** on held-out sentences, Zbor matches **52.5%** of Aromanian words to the right Romanian word, against **22.5%** for the "looks like Romanian" rule (2.3×). On single dictionary words it gets 20.5% vs 4.7% (4.4×), and on a new, harder folk-tale test set 37.9% vs 19.3% (2.0×). Green words are right **84%** of the time. The scoring is clipped, so a cheating "always *și*" model scores only about 5%.

**Second language (section 16):** Romanians also mix up Aromanian with **Romani**, an unrelated language. **Lav** (*lav* = "word" in Romani) uses the same method on two Romani Bibles (7,110 Kalderash + 7,157 Chergash verse pairs): 39.1% of words right on test verses vs 6.7% for "looks like Romanian", 43.3% vs 3.0% on everyday Romani spelled the way people type it online, and 42.9% vs 4.8% on Romani words Romanians know from songs. A small language detector tells Aromanian, Romani and Romanian apart (96.8%, 100% and 95.4% of test sentences).

**Website:** https://claude.ai/artifact/P6WQ4n2DAcC3ryvEY1Ap3y runs both helpers in the browser, using the file this notebook exports (section 18).

**If you only want to read:** every cell already shows its output, so you can read the results without running anything.

In [1]:
# 0. Setup
!pip -q install rapidfuzz gradio
import os, re, io, glob, math, zipfile, unicodedata, random, requests
from collections import Counter
from functools import lru_cache
from getpass import getpass
import numpy as np, pandas as pd
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein
from IPython.display import display, HTML
pd.set_option('display.max_colwidth', 100)
random.seed(42); np.random.seed(42)


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Our own small test sets

We wrote three small test files by hand from public sources (one source per line), so this notebook can test everyday language and not only the corpus. They are stored here, so the notebook runs on its own in Colab.

In [2]:
OUR_TEST_FILES = {
    'phrasebook_test.csv': r'''id,aro,ro,kind,verification,source
1,Bunã dzua,Bună ziua,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
2,Cum ti chiamã,Cum te cheamă,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
3,Di cãtsi anji esht,Câți ani ai,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
4,Cumu hits,Ce mai faceți,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
5,Cumu eshti,Ce mai faci,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
6,Tsi fats,Ce faci,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
7,S'nã videmu cu ghine,La revedere,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
8,S'nã avdzãmu ghiniatsa,Numai bine,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
9,Vã plãcãrsescu,Vă rog,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
10,S'mi hãrãdzesht,Iartă-mă,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
11,Haristo,Mulțumesc,word,V-open,https://en.wikipedia.org/wiki/Aromanian_language
12,Ye,Da,word,V-open,https://en.wikipedia.org/wiki/Aromanian_language
13,Nu,Nu,word,V-open,https://en.wikipedia.org/wiki/Aromanian_language
14,Nu adukiescu,Nu înțeleg,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
15,Nu shtiu,Nu știu,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
16,Yu easti toaletlu,Unde este toaleta,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
17,Zburats anglikiashti,Vorbiți englezește,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
18,Mini escu un student,Eu sunt student,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
19,Mini escu un om bun,Eu sunt un om bun,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
20,Eshti mushat,Ești frumos,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
21,armãn,aromân,word,V-open,https://en.wikipedia.org/wiki/Aromanian_language
22,limba armãneascã,limba aromână,phrase,V-open,https://en.wikipedia.org/wiki/Aromanian_language
23,Ti multsã-anji Pashtili,La mulți ani de Paști,phrase,V-open,https://www.g4media.ro/cum-se-spune-paste-fericit-si-hristos-a-inviat-in-alte-limbi.html
24,Hristolu-nye,Hristos a înviat,word,V-open (variant: V-snippet),https://www.g4media.ro/cum-se-spune-paste-fericit-si-hristos-a-inviat-in-alte-limbi.html
25,Ti multsã-anji,La mulți ani,phrase,V-snippet,https://www.omniglot.com/language/phrases/aromanian.htm
26,Ghini vinit,Bine ați venit,phrase,V-snippet,https://www.omniglot.com/language/phrases/aromanian.htm
27,Ghini escu,Sunt bine,phrase,V-snippet,https://www.omniglot.com/language/phrases/aromanian.htm
28,zbor,cuvânt,word,V-open,https://en.wiktionary.org/wiki/zbor
29,zburãscu,"vorbi, vorbesc",word,V-open,https://en.wiktionary.org/wiki/zbor
30,cuvendã,cuvânt,word,V-open,https://en.wiktionary.org/wiki/zbor
31,vãrnu,vreun,word,V-open,https://en.wiktionary.org/wiki/v%C3%A3rnu
32,aoa,aici,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
33,adzã,azi,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
34,aeri,ieri,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
35,acasã,acasă,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
36,afoarã,afară,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
37,ai,hai,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
38,apã,apă,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
39,agiungu,"ajunge,  sosi",word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
40,afugu,pleca,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
41,alcã,smântână,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
42,babã,bunică,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
43,aharistisit,mulțumit,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
44,avlii,curte,word,V-open (community),https://roa-rup.wikipedia.org/wiki/Dic%C8%9Bionar_arom%C3%A2n%E2%80%93rom%C3%A2n
45,cusurin,văr,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
46,dimãndare,poruncă,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
47,fumealje,"familie, copii",word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
48,largu,departe,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
49,mur,zid,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
50,mes,lună,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
51,etã,"vârstă, secol",word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
52,agiun,flămând,word,V-open,https://ro.wikipedia.org/wiki/Limba_arom%C3%A2n%C4%83
53,ghine,bine,word,V-open,https://curentul.net/2021/11/05/despre-aromani-si-romani-limba-si-identitate/
54,gios,jos,word,V-open,https://curentul.net/2021/11/05/despre-aromani-si-romani-limba-si-identitate/
55,vruta mea,iubita mea,phrase,V-open,https://curentul.net/2021/11/05/despre-aromani-si-romani-limba-si-identitate/''',
    'romani_phrasebook_test.csv': r'''id,rom,rom_std,ro,kind,source
2,Dobroi tuca,Dobroj tuke,bună ziua,phrase,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
3,Socares?,So keres?,Ce faci?,phrase,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
4,Socares prala?,"So keres, phrala?","Ce faci, frate?",phrase,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
5,So halean?,So xalean?,Ce ai mâncat?,phrase,https://dilemaveche.ro/sectiune/tema-saptamanii/cuvinte-tiganesti-in-limba-romana-625165.html
8,Mai bes hari,Maj beš xanci,Mai stai puțin,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
11,Haida jas manta ando foro?,"Haide, džas manţa ando foro?",Mergi cu mine în oraș?,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
13,Camis te jas manta?,Kames te džas manţa?,Vrei să mergi cu mine?,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
14,Hai vi tu avri,Haj vi tu avri!,Hai și tu afară,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
17,Haida care,Haide khere,Hai acasă,phrase,https://dilemaveche.ro/sectiune/tema-saptamanii/cuvinte-tiganesti-in-limba-romana-625165.html
18,Sar busos?,Sar bušos?,Cum te cheamă?,phrase,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
19,Chechi bars si tut?,Keci berš si tut?,Câți ani ai?,phrase,https://blogg.sh.se/romstuduni/wp-content/uploads/2022/01/klic-SCR-Eng-01.pdf
21,Na dara,Na dara!,Nu te teme,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
22,Cai san?,Kaj san?,Unde ești?,phrase,https://en.wiktionary.org/wiki/sar_san
27,Chechi si o ceaso?,Keci si o ceaso?,Cât e ceasul?,phrase,https://blogg.sh.se/romstuduni/wp-content/uploads/2022/01/klic-SCR-Eng-01.pdf
28,Sukar san,Šukar san,Ești frumos,phrase,https://dexonline.ro/definitie/%C8%99ucar
51,Te trais but,Te traies but,Să trăiești mult,phrase,https://www2.arnes.si/~eusmith/Romany/phrases.html
90,Sar san?,Sar san?,Cum ești?,phrase,https://en.wiktionary.org/wiki/sar_san
9,mishto,mišto,bine,word,https://dexonline.ro/definitie/mi%C8%99to
32,seav,šav,copil,word,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
36,jucal,džukel,câine,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
38,care,khere,acasă,word,https://dilemaveche.ro/sectiune/tema-saptamanii/cuvinte-tiganesti-in-limba-romana-625165.html
39,ja,dža,du-te,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
46,bari,baro,mare,word,https://dilemaveche.ro/sectiune/tema-saptamanii/cuvinte-tiganesti-in-limba-romana-625165.html
52,pralo,phral,frate,word,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
53,moro,miro,meu,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
54,devla,Devla,Doamne,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
57,shukar,šukar,frumos,word,https://dexonline.ro/definitie/%C8%99ucar
58,san,san,ești,word,https://en.wiktionary.org/wiki/sar_san
62,cai,kaj,unde,word,https://en.wiktionary.org/wiki/sar_san
63,so,so,ce,word,https://en.wiktionary.org/wiki/Category:Romani_phrasebook
64,kames,kames,vrei,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
65,beshes,bešes,stai,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
67,manta,manţa,cu mine,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
69,sosoi,šošoj,iepure,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
71,jas,džas,mergi,word,https://www2.arnes.si/~eusmith/Romany/phrases.html
72,chechi,keci,câți,word,https://blogg.sh.se/romstuduni/wp-content/uploads/2022/01/klic-SCR-Eng-01.pdf
91,lovo,lovo,bani,word,https://dexonline.ro/definitie/lovele
92,baro,baro,mare,word,https://dexonline.ro/definitie/barosan''',
    'romani_songs_test.csv': r'''id,rom,rom_std,ro,kind,heard_in,source
1,Chaiorie,Čhajorije,fetițo,word,"song title: Ionuț Cercel, ""Chaiorie""; Sofi Marinova, ""Chaiorie Shukarie""",https://rm.coe.int/factsheets-on-romani-culture-1-6-romani-given-names/1680aac36a
2,Chaiorie shukarie,Čhajorije šukarije,fetiță frumoasă,phrase,"song title: Sofi Marinova, ""Chaiorie Shukarie""",https://en.wiktionary.org/wiki/%C4%87hajorri
3,Tu san shukar,Tu san šukar,tu ești frumoasă,phrase,"common in manele and Roma songs",https://dexonline.ro/definitie/%C8%99ucar
4,shukar,šukar,frumos,word,"common in manele and Roma songs",https://dexonline.ro/definitie/%C8%99ucar
5,Kamav tut,Kamav tut,te iubesc,phrase,"common in Roma love songs",https://www.reddit.com/r/romani/comments/u0ujlp/how_to_say_i_love_you/
6,Bahtale Romale,Baxtale Romale,romi norocoși,phrase,"song: Gipsy Casual, ""Romale""; Bahtalo band, ""Bahtalo Romale""",https://en.wikipedia.org/wiki/Romani_anthem
7,Gelem gelem,Gelem gelem,am mers am mers,phrase,"the Romani anthem, ""Gelem, Gelem""",https://en.wikipedia.org/wiki/Romani_anthem
8,Loli phabaj,Loli phabaj,măr roșu,phrase,"song title: Esma Redžepova, ""Loli phabaj""",https://www.youtube.com/watch?v=TmhA1993LWY
9,Devla,Devla,Doamne,word,"common in manele and Roma songs",https://www2.arnes.si/~eusmith/Romany/phrases.html
10,mishto,mišto,bine,word,"common in manele; became Romanian slang",https://dexonline.ro/definitie/mi%C8%99to
11,Te aves bahtalo,Te aves baxtalo,să fii norocos,phrase,"common wish in Roma songs",https://www2.arnes.si/~eusmith/Romany/phrases.html
12,romale,Romale,romilor,word,"song: Gipsy Casual, ""Romale""",https://en.wikipedia.org/wiki/Romani_anthem''',
}
for name, text in OUR_TEST_FILES.items():
    if not os.path.exists(name):
        with open(name, 'w', encoding='utf-8') as f:
            f.write(text + '\n')
print({name: len(text.splitlines()) - 1 for name, text in OUR_TEST_FILES.items()}, 'rows')

{'phrasebook_test.csv': 55, 'romani_phrasebook_test.csv': 38, 'romani_songs_test.csv': 12} rows


## 1. Load the main Romanian–Aromanian parallel corpus

We use `aronlp/aromanian-romanian-MT-corpus-limited` (University of Bucharest; Petrariu & Nisioi 2024, Jerpelea et al. 2025): about 77,000 Romanian–Aromanian pairs, licence CC BY-NC 4.0. It has the Romanian text (`ron`), the Aromanian text in two spellings (`rup_cunia`, `rup_diaro`) and the `source` of each pair. We use the Cunia spelling.

The dataset is gated: open its page on Hugging Face and click **Agree**. Then either put the downloaded CSV next to this notebook (in Colab: the Files panel), or paste a **read** token from huggingface.co/settings/tokens when asked (or save it as a Colab secret called `HF_TOKEN`).

In [3]:
local = sorted(glob.glob('*.csv') + glob.glob('/content/*.csv'))
local = [f for f in local if 'aromanian-romanian' in f]
if local:
    print('Loading local file:', local[0])
    raw = pd.read_csv(local[0])
else:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
    login(token=token or getpass('Hugging Face read token: '))
    from datasets import load_dataset
    raw = load_dataset('aronlp/aromanian-romanian-MT-corpus-limited', split='train').to_pandas()
print(raw.shape, '| columns:', list(raw.columns))
ARO_COL, RO_COL = 'rup_cunia', 'ron'

Loading local file: aromanian-romanian-MT-corpus-limited.csv
(77210, 6) | columns: ['Unnamed: 0', 'ron', 'rup_cunia', 'rup_diaro', 'source', 'split']


## 2. Look at the data first

Before training anything, we read some pairs ourselves. About 40% of the corpus is Bible text, so words like *sabie* (sword) and *cetate* (citadel) are common, while modern words are rare. About a third of it is single dictionary entries (`dictionar_cuvinte`), and some is poetry. A few pairs have scanning errors (an *I* read as an *l*).

In [4]:
display(raw['source'].value_counts())

CEDILLA = str.maketrans({'ş': 'ș', 'Ş': 'Ș', 'ţ': 'ț', 'Ţ': 'Ț'})
WORD = re.compile(r'[^\W\d_]+')

def strip(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def tok(s):
    return WORD.findall(unicodedata.normalize('NFC', str(s)).translate(CEDILLA).lower())

def prepare(d, maxlen=40):
    d = d[(d.aro.str.strip() != '') & (d.ro.str.strip() != '')].copy()
    d['aro_t'], d['ro_t'] = d.aro.map(tok), d.ro.map(tok)
    return d[d.aro_t.str.len().between(1, maxlen) & d.ro_t.str.len().between(1, maxlen)]

df = raw[[ARO_COL, RO_COL, 'source']].dropna().astype(str)
df.columns = ['aro', 'ro', 'source']
df = prepare(df.drop_duplicates('aro'))
print(f'{len(df):,} clean pairs')
for _, r in df.sample(4, random_state=1).iterrows():
    print(f'[{r.source}] ARO: {r.aro[:110]}\n{" " * (len(r.source) + 3)}RO:  {r.ro[:110]}\n')

source
bible                        30535
dictionar_cuvinte            27310
eroi_patati                   6931
tom_sawyer                    3134
george_vranea                 2252
divina_comedie                2243
dictionar_expresii            2154
un_veac_de_poezie_aromana     1765
micul_print                    623
tao_tza                        263
Name: count, dtype: int64

65,708 clean pairs
[dictionar_cuvinte] ARO: fundutosu
                    RO:  stufos

[bible] ARO: Sã spitrunda nauntru tu omlu shi-ahanda tu inima-a lui.
        RO:  Ca să pătrundă înlăuntrul omului și în adâncimea inimii lui.

[bible] ARO: Shi parintsalji a ljei, cara sh-eara-ndreptsa, u-nvitsara hiljea-a lor dupa nomlu-al Moise.
        RO:  Și părinții ei, drepți fiind, au învățat pe fiica lor după legea lui Moise.

[bible] ARO: Ca unã njilji di-anji dininti-a ocljilor a Tãlji suntu ca dzutia di-aseara, tsi tricu sh-ca cãrãulea-a noaptil
        RO:  Că o mie de ani înaintea ochilor Tăi sunt ca ziua de ieri, care a trecut și ca straja nopții.



## 3. Extra data we found: Aromanian folk tales

The "limited" corpus leaves out one source: a collection of Aromanian **folk tales** recorded in the 1970s, with Romanian translations. The same research group publishes these on GitHub ([senisioi/aromanian](https://github.com/senisioi/aromanian), GPL-3.0). Folk tales are everyday narrative language, very different from the Bible text, so they are useful in two ways:
- **extra training data:** the first half of the tales;
- **a harder, more realistic test:** the last quarter of the tales, which no setting is ever tuned on (the quarter in between is for tuning).

We keep the tales in story order, so a story is never split between training and testing, and we drop the few sentences that also appear in the main corpus.

In [5]:
TALES_URL = 'https://github.com/senisioi/aromanian/raw/main/unsplit.zip'
z = zipfile.ZipFile(io.BytesIO(requests.get(TALES_URL, timeout=120).content))
read = lambda suffix: z.read(next(n for n in z.namelist() if n.endswith(suffix))).decode('utf-8').split('\n')
tales = pd.DataFrame({'aro': read('corpus.rup_cun'), 'ro': read('corpus.ro')})
tales = prepare(tales).reset_index(drop=True)
key = lambda s: re.sub(r'\W+', ' ', unicodedata.normalize('NFC', str(s)).lower()).strip()
seen_ro = set(df.ro.map(key))
tales = tales[~tales.ro.map(key).isin(seen_ro)].reset_index(drop=True)
n_t = len(tales)
tales_train, tales_dev, tales_test = tales[:n_t // 2], tales[n_t // 2: 3 * n_t // 4], tales[3 * n_t // 4:]
print(f'{n_t:,} folk-tale pairs: {len(tales_train)} train | {len(tales_dev)} dev | {len(tales_test)} test')
print('Example:', tales.aro.iloc[2], '→', tales.ro.iloc[2])

1,474 folk-tale pairs: 737 train | 368 dev | 369 test
Example: Eara un lup, - Masturlu-Nicola, shi unã vulpe, - Chira-Mara. → A fost odată un lup, - Cumătrul-Nicola, și-o vulpe, - Cumătra-Mara.


## 4. Split the main corpus into training, dev and test sets

90% for training, 5% **dev** (used to compare ideas and choose settings), 5% **test** (used only once, for the final results). From dev and test we build two kinds of evaluation sets:
- **sentences:** pairs with at least 3 Aromanian words;
- **dictionary words:** single-word entries from the corpus's dictionary part, where each Aromanian word has a known Romanian meaning. This is a clean word-level test.

In [6]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df)
test, dev, train = df[:int(.05 * n)], df[int(.05 * n):int(.10 * n)], df[int(.10 * n):]

def eval_sets(part, tales_part):
    return {'sentences': part[part.aro_t.str.len() >= 3],
            'dictionary words': part[part.source.str.startswith('dictionar_cuvinte') & (part.aro_t.str.len() == 1)],
            'folk tales': tales_part}
DEV, TEST = eval_sets(dev, tales_dev), eval_sets(test, tales_test)
train_plus = pd.concat([train, tales_train])
print(f'train {len(train):,} pairs (+{len(tales_train)} folk-tale pairs)')
print({k: len(v) for k, v in DEV.items()}, '<- dev sets')
print({k: len(v) for k, v in TEST.items()}, '<- test sets')

train 59,138 pairs (+737 folk-tale pairs)
{'sentences': 2268, 'dictionary words': 880, 'folk tales': 368} <- dev sets
{'sentences': 2284, 'dictionary words': 868, 'folk tales': 369} <- test sets


## 5. How we score: clipped word matching

For each Aromanian word, a method guesses one Romanian word. The guess counts as a **hit** if that word appears in the correct Romanian translation. Each word of the correct translation can be matched **only once** (clipped matching, the same idea as BLEU).

We learned why clipping matters the hard way. Our first version didn't clip, and a "model" that answers *și* for every word scored 67%, higher than our real model, because *și* appears in almost every sentence. With clipping, that trick scores about 5%.

We report:
- **exact:** the same word form;
- **lenient:** the same first 5 letters, ignoring diacritics (so *casa* and *casă*, or *omul* and *omului*, count);
- **content words:** exact hits for Aromanian words with 4+ letters only, where the filler words don't help;
- **unseen words:** lenient hits for Aromanian words that never appeared in training.

In [7]:
def evaluate(split, guess, counts=None, src_col='aro_t'):
    counts = counts or {}
    t = Counter()
    for a_toks, r_toks in zip(split[src_col], split.ro_t):
        ref, stems = Counter(r_toks), Counter(strip(w)[:5] for w in r_toks)
        for w in a_toks:
            g = guess(w)
            s5 = strip(g)[:5]
            hit = ref[g] > 0
            soft = hit or stems[s5] > 0
            if hit: ref[g] -= 1
            if soft: stems[s5] -= 1
            t['n'] += 1; t['exact'] += hit; t['lenient'] += soft
            if len(w) >= 4: t['cn'] += 1; t['content'] += hit
            if counts.get(w, 0) == 0: t['un'] += 1; t['unseen'] += soft
    pct = lambda a, b: round(100 * t[a] / t[b], 1) if t[b] else None
    return {'exact': pct('exact', 'n'), 'lenient': pct('lenient', 'n'),
            'content words': pct('content', 'cn'), 'unseen words': pct('unseen', 'un')}

## 6. Baselines

- **always the most common word:** the cheating "model" from section 5;
- **copy:** leave the Aromanian word unchanged;
- **looks like Romanian:** what a Romanian reader does in their head. Turn common Aromanian spellings into Romanian ones (*sh* → *ș*, *ts* → *ț*, *ã* → *ă*), then pick the most similar Romanian word.

In [8]:
ro_counts_all = Counter(w for s in train_plus.ro_t for w in s)
TOP_RO = ro_counts_all.most_common(1)[0][0]
ARO2RO = [('sh', 'ș'), ('ts', 'ț'), ('dz', 'z'), ('lj', 'li'), ('nj', 'ni'), ('ã', 'ă'), ('ľ', 'l'), ('ń', 'n'), ('ḑ', 'z')]
def romanize(w):
    for a, b in ARO2RO:
        w = w.replace(a, b)
    return w

RO_VOCAB = [w for w, c in ro_counts_all.items() if c >= 2]
RO_KEYS = [strip(w) for w in RO_VOCAB]

@lru_cache(maxsize=None)
def looks_like(w):
    best = process.extractOne(strip(romanize(w)), RO_KEYS, scorer=Levenshtein.normalized_similarity)
    return RO_VOCAB[best[2]], best[1]

BASELINES = {f'always "{TOP_RO}"': lambda w: TOP_RO,
             'copy word': lambda w: w,
             'looks like Romanian': lambda w: looks_like(w)[0]}
for w in ['zbor', 'casã', 'shi', 'omlu']:
    print(w, '-> looks like', looks_like(w))

zbor -> looks like ('zbor', 1.0)
casã -> looks like ('casa', 1.0)
shi -> looks like ('și', 1.0)
omlu -> looks like ('omului', 0.6666666666666667)


## 7. Our model: learn word meanings from the sentence pairs

We use **IBM Model 1**, a classic word-alignment method. The idea: if an Aromanian word keeps appearing in sentences whose Romanian translation contains a certain word, they probably mean the same thing. The model starts by assuming every pairing is equally likely, then repeats two steps (expectation–maximisation):
1. In each sentence pair, share out each Romanian word between the Aromanian words, according to the current probabilities.
2. Re-count how often each Aromanian word was paired with each Romanian word, and update the probabilities.

The result is P(Romanian word | Aromanian word) for every pair of words seen together. For each Aromanian word we keep the 3 most likely meanings; the probability of the best one is its **score**.

For words Zbor has never seen, it tries two fallbacks:
1. **Spelling bridge:** converts other Aromanian spellings (*ș, ț, ľ, ń, ă, â*) to the corpus's Cunia spelling (*sh, ts, lj, nj, ã*).
2. **Similar known word:** borrows the meaning of the most similar Aromanian word it does know (spelling similarity ≥ 0.75). This often catches another form of the same word, e.g. a plural.

In [9]:
def ibm1(src_sents, tgt_sents, src_stoi, tgt_stoi, iters=6, diag=0.0):
    """Returns a table src, tgt, p = P(tgt word | src word).
    diag > 0 adds a position prior (as in fast_align): a word pair counts more when the two words sit at a similar
    relative position in their sentences, weight exp(-diag * |i/m - j/n|). Aromanian, Romani and Romanian mostly keep
    the same word order, so this is a reasonable hint."""
    V = len(tgt_stoi)
    keys, groups, wts, g = [], [], [], 0
    for s, t in zip(src_sents, tgt_sents):
        si = np.array([src_stoi[w] for w in s if w in src_stoi], dtype=np.int64)
        ti = np.array([tgt_stoi[w] for w in t if w in tgt_stoi], dtype=np.int64)
        if len(si) == 0 or len(ti) == 0:
            continue
        m, n = len(si), len(ti)
        keys.append(np.repeat(si, n) * V + np.tile(ti, m))   # every (src, tgt) pair in the sentence
        groups.append(np.tile(np.arange(g, g + n), m))      # which target position it belongs to
        wts.append(np.exp(-diag * np.abs((np.repeat(np.arange(m), n) + .5) / m - (np.tile(np.arange(n), m) + .5) / n)))
        g += n
    keys, groups, wts = np.concatenate(keys), np.concatenate(groups), np.concatenate(wts)
    uniq, pair_idx = np.unique(keys, return_inverse=True)
    pair_idx = pair_idx.ravel()
    src_of, tgt_of = uniq // V, uniq % V
    p = np.ones(len(uniq))
    for _ in range(iters):
        pv = p[pair_idx] * wts
        share = pv / np.bincount(groups, weights=pv, minlength=g)[groups]               # step 1
        counts = np.bincount(pair_idx, weights=share, minlength=len(uniq))
        p = counts / np.bincount(src_of, weights=counts, minlength=len(src_stoi))[src_of]  # step 2
    return pd.DataFrame({'src': src_of, 'tgt': tgt_of, 'p': p})

def train_model(tr, both_directions=False, topk=5, diag=0.0):
    ac = Counter(w for s in tr.aro_t for w in s)
    rc = Counter(w for s in tr.ro_t for w in s)
    a_itos, r_itos = list(ac), list(rc)
    a_stoi, r_stoi = {w: i for i, w in enumerate(a_itos)}, {w: i for i, w in enumerate(r_itos)}
    pairs = ibm1(tr.aro_t, tr.ro_t, a_stoi, r_stoi, diag=diag)
    pairs.columns = ['a', 'r', 'score']
    if both_directions:   # our first version: geometric mean of both directions
        back = ibm1(tr.ro_t, tr.aro_t, r_stoi, a_stoi)
        back.columns = ['r', 'a', 'p_back']
        pairs = pairs.merge(back, on=['a', 'r'])
        pairs['score'] = np.sqrt(pairs.score * pairs.p_back)
    top = pairs.sort_values('score', ascending=False).groupby('a').head(topk)
    gloss = {}
    for a, r, s in zip(top.a, top.r, top.score):
        gloss.setdefault(a_itos[a], []).append((r_itos[r], float(s)))
    return gloss, ac

DIARO2CUNIA = [('d̦', 'dz'), ('ḑ', 'dz'), ('ș', 'sh'), ('ț', 'ts'), ('ľ', 'lj'), ('ń', 'nj'), ('ă', 'ã'), ('â', 'ã')]
def to_cunia(w):
    for a, b in DIARO2CUNIA:
        w = w.replace(a, b)
    return w

def make_lookup(gloss, similar=0.75, bridge=True):
    known = list(gloss)
    @lru_cache(maxsize=None)
    def lookup(w):
        """Returns (dictionary key, how it was found) or (None, None)."""
        if w in gloss:
            return w, 'known'
        if bridge and to_cunia(w) in gloss:
            return to_cunia(w), 'spelling'
        if similar:
            m = process.extractOne(w, known, scorer=Levenshtein.normalized_similarity, score_cutoff=similar)
            if m:
                return m[0], 'similar'
        return None, None
    return lookup

def make_guess(gloss, lookup):
    def guess(w):
        k, _ = lookup(w)
        return gloss[k][0][0] if k else looks_like(w)[0]
    return guess

## 8. Experiments: which ideas actually help?

We compared five versions on the **dev** sets. Each row adds one idea to the row above, so we can see what each idea is worth.
- **A:** our first version, which scores each pair using both directions (Aromanian→Romanian and back).
- **B:** score with Aromanian→Romanian only. With both directions, rare Romanian words got unfairly high scores.
- **C:** B, plus borrowing the meaning of a similar known word when a word is unseen.
- **D:** C, plus the folk tales in training.
- **E:** D, plus a position prior: a word pair counts more when both words sit at a similar place in their sentences (the idea behind fast_align, Dyer et al. 2013). We tried strengths 2, 4 and 8 on dev; all helped a little, and we use 4, fast_align's usual value.

We also tried more training rounds and a "prefer meanings spelled like the Aromanian word" rule. Both made the dev scores slightly *worse*, so we dropped them.

In [10]:
CONFIGS = [('A: both directions (first version)', train, True, 0, 0.0),
           ('B: Aromanian→Romanian only', train, False, 0, 0.0),
           ('C: B + similar-word fallback', train, False, 0.75, 0.0),
           ('D: C + folk tales in training', train_plus, False, 0.75, 0.0),
           ('E: D + position prior (final)', train_plus, False, 0.75, 4.0)]
rows = []
for name, tr, both, sim, diag in CONFIGS:
    gloss, counts = train_model(tr, both_directions=both, diag=diag)
    guess = make_guess(gloss, make_lookup(gloss, similar=sim, bridge=False))
    row = {'version': name}
    for set_name, part in DEV.items():
        r = evaluate(part, guess, counts)
        row[f'{set_name}: exact'] = r['exact']
        if set_name == 'sentences':
            row['sentences: content words'] = r['content words']
            row['sentences: unseen words'] = r['unseen words']
    rows.append(row)
    print('done:', name)
ablation = pd.DataFrame(rows).set_index('version')
display(ablation)

# The last version (E) is our final model
GLOSS, aro_counts = gloss, counts
lookup = make_lookup(GLOSS, similar=0.75, bridge=True)
zbor_guess = make_guess(GLOSS, lookup)
print(f'Final dictionary: {len(GLOSS):,} Aromanian words')
for w in ['zbor', 'shi', 'casã', 'omlu', 'coarda']:
    print(w, '->', GLOSS.get(w))

done: A: both directions (first version)


done: B: Aromanian→Romanian only


done: C: B + similar-word fallback


done: D: C + folk tales in training


done: E: D + position prior (final)


,sentences: exact,sentences: content words,sentences: unseen words,dictionary words: exact,folk tales: exact
version,,,,,
A: both directions (first version),48.9,50.2,14.9,6.0,34.0
B: Aromanian→Romanian only,51.3,53.8,14.9,6.1,38.7
C: B + similar-word fallback,51.4,54.1,25.0,22.6,39.0
D: C + folk tales in training,51.5,54.2,25.5,22.7,41.3
E: D + position prior (final),51.8,54.8,27.6,22.8,41.3


Final dictionary: 48,632 Aromanian words
zbor -> [('cuvânt', 0.4540279790285527), ('vorbă', 0.18714538304922843), ('cuvântul', 0.07658068710587036), ('o', 0.03487307431956433), ('îi', 0.018924840391212342)]
shi -> [('și', 0.904892718496333), ('iar', 0.041781935228958046), ('a', 0.03204213543322729), ('pe', 0.006623729726211243), ('de', 0.0031042387854783157)]
casã -> [('casă', 0.6800028043148807), ('casa', 0.041188543785988886), ('în', 0.030818870932901397), ('o', 0.029018047865978124), ('familie', 0.025109206602810612)]
omlu -> [('omul', 0.8449986571636828), ('om', 0.08910006711642585), ('pe', 0.022781241204211527), ('omului', 0.021636711693782555), ('care', 0.005936657650251878)]
coarda -> [('sabia', 0.8305751628809852), ('sabie', 0.13956370283037753), ('spada', 0.0077031053276591065), ('intre', 0.0059441245249018705), ('cu', 0.003367767670808313)]


### Data check: pairs that look wrong or contradict each other

We wrote a checker that looks through the **training** pairs for problems:
- **length mismatch:** one side is more than 3 times longer than the other (often a scanning error or a note instead of a translation);
- **number mismatch:** the two sides contain different numbers (often page numbers inside the text);
- **untranslated:** the Romanian side is just a copy of the Aromanian side;
- **conflicting dictionary entries:** the same Aromanian dictionary word with two different Romanian meanings. Many of these are real double meanings (*cari* = "cine" or "când"), but some are errors.

Then we test, on **dev**, whether training without the flagged pairs helps. Research on corpus cleaning (Khayrallah & Koehn 2018) found that word-statistics models like ours are hurt much less by noisy pairs than neural translators, so we expect little change. **Result:** sentences and folk tales stay the same and dictionary words move by 0.4 points (about two words), so we keep all pairs and report the flags instead.

In [11]:
la, lr = train_plus.aro_t.str.len(), train_plus.ro_t.str.len()
ratio = (la + 1) / (lr + 1)
nums = lambda s: tuple(sorted(re.findall(r'\d+', s)))
checks = pd.DataFrame({
    'length mismatch': (ratio > 3) | (ratio < 1 / 3),
    'number mismatch': train_plus.aro.map(nums) != train_plus.ro.map(nums),
    'untranslated': (train_plus.aro.str.lower().str.strip() == train_plus.ro.str.lower().str.strip()) & (la > 3)})
single = train_plus[(la == 1) & (lr <= 2)]
meanings = {}
for a, r in zip(single.aro_t, single.ro_t):
    meanings.setdefault(a[0], set()).add(' '.join(r))
conflicts = {w: sorted(m) for w, m in meanings.items() if len(m) > 1}
print('Flagged training pairs:', checks.sum().to_dict(), '| any:', int(checks.any(axis=1).sum()), f'of {len(train_plus):,}')
print(f'Dictionary words with conflicting meanings: {len(conflicts)}, e.g.', list(conflicts.items())[:6])
for name in checks.columns:
    ex = train_plus[checks[name]].head(2)
    for a, r in zip(ex.aro, ex.ro):
        print(f'  [{name}] {a[:70]}  →  {r[:70]}')

clean = train_plus[~checks.any(axis=1)]
g_clean, c_clean = train_model(clean, diag=4.0)
f_clean = make_guess(g_clean, make_lookup(g_clean, similar=0.75, bridge=False))
f_final = make_guess(GLOSS, make_lookup(GLOSS, similar=0.75, bridge=False))
print('\nDEV, exact matches (%):')
display(pd.DataFrame({'all training pairs (final)': {s: evaluate(p, f_final, aro_counts)['exact'] for s, p in DEV.items()},
                      'without flagged pairs': {s: evaluate(p, f_clean, c_clean)['exact'] for s, p in DEV.items()}}).T)
print('If removing the flagged pairs does not help on dev, we keep all pairs: the model already outweighs a few bad ones.')

Flagged training pairs: {'length mismatch': 110, 'number mismatch': 475, 'untranslated': 3} | any: 579 of 59,875
Dictionary words with conflicting meanings: 69, e.g. [('angreacã', ['atârna greu', 'atârnă']), ('martsu', ['martie', 'marț']), ('alithea', ['adevăr', 'adevărul']), ('e', ['da', 'o', 'sau']), ('nu', ['aș', 'nici', 'nu']), ('hiljiu', ['fiu', 'fiule'])]
  [length mismatch] Cipu  →  Grămustean din Bulgaria-poreclă data de fârșiroți
  [length mismatch] Shi-atumtsea  →  Și atunci toți bărbații destoinici de luptă ai lui Israel s-au repezit
  [number mismatch] Dukenjli nu arnjudzescu na lãngoari, dukenjli suntu mplini di oaminj t  →  Prăvăliile nu miros a odaie de bolnav – și sunt pline de lucruri pe ca
  [number mismatch] "O, domnul a nostu!  →  "0, domnul nostru!
  [untranslated] Ira din Iatir, Gareb din Iatir;  →  Ira din Iatir, Gareb din Iatir;
  [untranslated] Salu, Amoc, Hilchia, Iedaia.  →  Salu, Amoc, Hilchia, Iedaia.



DEV, exact matches (%):


,sentences,dictionary words,folk tales
all training pairs (final),51.8,22.8,41.3
without flagged pairs,51.8,23.2,41.3


If removing the flagged pairs does not help on dev, we keep all pairs: the model already outweighs a few bad ones.


## 9. Final results on the test sets (used only once)

In [12]:
methods = dict(BASELINES)
methods['Zbor (ours)'] = zbor_guess
final = []
for name, f in methods.items():
    row = {'method': name}
    for set_name, part in TEST.items():
        r = evaluate(part, f, aro_counts)
        row[f'{set_name}: exact'] = r['exact']
        if set_name == 'sentences':
            row['sentences: lenient'] = r['lenient']
            row['sentences: content words'] = r['content words']
    final.append(row)
final = pd.DataFrame(final).set_index('method')
print('TEST: % of Aromanian words whose Romanian guess matches the correct translation (clipped)')
display(final)

def band(w):
    c = aro_counts.get(w, 0)
    return 'unseen' if c == 0 else 'rare 1-4' if c < 5 else 'medium 5-49' if c < 50 else 'common 50+'
by_band = {}
for b in ['common 50+', 'medium 5-49', 'rare 1-4', 'unseen']:
    sub = TEST['sentences'].copy()
    sub['aro_t'] = sub.aro_t.map(lambda ws: [w for w in ws if band(w) == b])
    by_band[b] = {name: evaluate(sub, f)['lenient'] for name, f in methods.items() if not name.startswith('always')}
print('\nTEST sentences, lenient, by how often the Aromanian word appeared in training:')
display(pd.DataFrame(by_band).T)

TEST: % of Aromanian words whose Romanian guess matches the correct translation (clipped)


,sentences: exact,sentences: lenient,sentences: content words,dictionary words: exact,folk tales: exact
method,,,,,
"always ""și""",5.3,5.3,4.9,0.1,5.0
copy word,12.9,17.6,8.2,0.2,12.0
looks like Romanian,22.5,26.2,16.6,4.7,19.3
Zbor (ours),52.5,56.0,55.6,20.5,37.9



TEST sentences, lenient, by how often the Aromanian word appeared in training:


,copy word,looks like Romanian,Zbor (ours)
common 50+,17.8,26.9,55.5
medium 5-49,16.6,25.8,65.8
rare 1-4,15.3,20.9,51.3
unseen,18.6,16.2,28.4


In [13]:
z, lk = final.loc['Zbor (ours)'], final.loc['looks like Romanian']
print('SUMMARY (test sets, exact clipped matches):')
for s in TEST:
    print(f'  {s:17s} Zbor {z[f"{s}: exact"]:5.1f}%  vs  looks-like {lk[f"{s}: exact"]:5.1f}%  ({z[f"{s}: exact"] / lk[f"{s}: exact"]:.1f}x)')

SUMMARY (test sets, exact clipped matches):
  sentences         Zbor  52.5%  vs  looks-like  22.5%  (2.3x)
  dictionary words  Zbor  20.5%  vs  looks-like   4.7%  (4.4x)
  folk tales        Zbor  37.9%  vs  looks-like  19.3%  (2.0x)


**What we see:**
- Reading Aromanian "as if it were Romanian" gets only about one word in five right. Zbor gets about twice as many right on sentences and folk tales, and about four times as many on single dictionary words.
- The cheating "always *și*" model scores only about 5%, so the clipped metric can't be gamed that way.
- The folk tales are much harder than the corpus sentences, because they use everyday words the Bible-heavy corpus rarely has. Adding half of them to training helped a lot on the other half (section 8).
- The similar-word fallback is what makes the dictionary-word test work, because most of those words never appear in the sentence data.
- Unseen words remain our weakest point.

## 10. Confidence colours that mean something

A colour is only useful if green really means "you can trust this". We pick the thresholds on the **dev** sentences:
- **green:** the lowest score where at least 80% of the guesses at or above it are right (lenient);
- **yellow:** from there down, as long as each 0.1-wide slice of scores is still at least 50% right;
- **red:** anything below, every word whose meaning was borrowed from a similar word, and every unknown word.

Then we check on the **test** sentences whether that still holds.

In [14]:
def scored(split):
    rows = []
    for a_toks, r_toks in zip(split.aro_t, split.ro_t):
        stems = Counter(strip(w)[:5] for w in r_toks)
        for w in a_toks:
            k, how = lookup(w)
            if k is None:
                continue
            g, s = GLOSS[k][0]
            s = s if how != 'similar' else 0.0      # borrowed meanings are never shown as "sure"
            hit = stems[strip(g)[:5]] > 0
            if hit:
                stems[strip(g)[:5]] -= 1
            rows.append((s, hit))
    return pd.DataFrame(rows, columns=['score', 'hit'])

GRID = np.round(np.arange(0.02, 0.96, 0.02), 2)
dev_rows = scored(DEV['sentences'])
GREEN = next((float(t) for t in GRID if (dev_rows.score >= t).sum() >= 50 and dev_rows[dev_rows.score >= t].hit.mean() >= 0.8), 0.8)
YELLOW = GREEN
for t in GRID[::-1]:
    if t >= GREEN:
        continue
    sl = dev_rows[(dev_rows.score >= t) & (dev_rows.score < t + 0.1)]
    if len(sl) >= 30 and sl.hit.mean() < 0.5:
        break
    YELLOW = float(t)
print(f'Thresholds chosen on dev: green >= {GREEN}, yellow >= {YELLOW}')

def colour(s):
    return 'green' if s >= GREEN else 'yellow' if s >= YELLOW else 'red'

test_rows = scored(TEST['sentences'])
test_rows['colour'] = test_rows.score.map(colour)
calib = test_rows.groupby('colour').hit.agg(['mean', 'size'])
calib['mean'] = (calib['mean'] * 100).round(1)
calib.columns = ['% right on TEST', 'words']
display(calib)

Thresholds chosen on dev: green >= 0.92, yellow >= 0.56


,% right on TEST,words
colour,,
green,84.1,3452
red,41.9,17652
yellow,65.6,15933


## 11. Look-alike traps

A trap is an Aromanian word that is spelled like a Romanian word, while our model is fairly confident (score ≥ 0.4) it means something else. We only keep a trap when:
- the real meaning isn't just another form of the same word (*anji* → *ani*);
- the real meaning isn't one of the 100 most common Romanian words. Those are usually alignment noise (*a*, *mai*, *pe*), not a real meaning;
- the look-alike Romanian word appears at least 5 times in the Romanian training text, so a reader might plausibly think of it;
- the Aromanian word appears at least 5 times in training.

These are **possible** traps: a native speaker still has to confirm them (section 15).

In [15]:
TRAP_MIN = 0.4
COMMON_RO = {w for w, _ in ro_counts_all.most_common(100)}
def same_word(x, y):
    x, y = strip(x), strip(y)
    return x[:3] == y[:3] or Levenshtein.normalized_similarity(x, y) >= 0.6

traps = []
for a, opts in GLOSS.items():
    if aro_counts[a] < 5 or len(a) < 3:
        continue
    r, s = opts[0]
    if s < TRAP_MIN or r in COMMON_RO or len(r) < 3:
        continue
    lookalike, sim = looks_like(a)
    if (sim < 0.8 or ro_counts_all[lookalike] < 5 or same_word(lookalike, r) or same_word(romanize(a), r)
            or any(same_word(x, lookalike) for x, _ in opts)):
        continue
    traps.append({'aromanian': a, 'looks like (ro)': lookalike, 'actually means (ro)': r,
                  'confidence': round(s, 2), 'times seen': aro_counts[a]})
TRAPS = pd.DataFrame(traps).sort_values('times seen', ascending=False).reset_index(drop=True)
TRAP = {r['aromanian']: (r['looks like (ro)'], r['actually means (ro)']) for _, r in TRAPS.iterrows()}
print(f'{len(TRAPS)} possible traps found. Is "zbor" one of them?', 'zbor' in TRAP)
print('\nSome traps we found especially interesting:')
display(TRAPS[TRAPS.aromanian.isin(['zbor', 'fara', 'vara', 'coarda', 'bana', 'asimi', 'boatsea', 'tora'])])
print('\nTop 30 by how often they appear:')
display(TRAPS.head(30))
TRAPS.to_csv('traps.csv', index=False)   # the list we send to an Aromanian speaker

177 possible traps found. Is "zbor" one of them? True

Some traps we found especially interesting:


,aromanian,looks like (ro),actually means (ro),confidence,times seen
1,tora,țora,acum,0.84,889
3,bana,banda,viața,0.52,535
5,fara,fără,neamul,0.59,323
7,boatsea,bătea,glasul,0.91,286
9,vara,vară,cineva,0.51,253
13,asimi,asini,argint,0.94,213
15,zbor,zbor,cuvânt,0.45,195
20,coarda,coardă,sabia,0.83,158



Top 30 by how often they appear:


,aromanian,looks like (ro),actually means (ro),confidence,times seen
0,ghini,ghici,însă,0.44,1236
1,tora,țora,acum,0.84,889
2,contra,contra,împotriva,0.80,693
3,bana,banda,viața,0.52,535
4,mash,mas,numai,0.75,435
5,fara,fără,neamul,0.59,323
6,fatsi,frați,face,0.47,318
7,boatsea,bătea,glasul,0.91,286
8,vasilje,vasili,rege,0.90,260
9,vara,vară,cineva,0.51,253


## 12. Adding context with a sentence translator (AroTranslate)

Our dictionary gives each word one meaning without looking at the sentence. That is why many simple words look unsure even when the meaning is right. *ghini* can mean "bine" or "însă", and a probability split between forms of one word (*noapte* / *noaptea*) never looks certain.

**AroTranslate** (Jerpelea, Rădoi & Nisioi, COLING 2025) is a neural translator for Aromanian: Meta's NLLB-200 model fine-tuned on about 100,000 Aromanian–Romanian pairs. We use its small CPU version (`aronlp/NLLB-rup-ron-eng-ct2`, CC BY-NC 4.0; gated: click *Agree* on its Hugging Face page).

We combine the two models:
1. AroTranslate translates the whole sentence.
2. For each Aromanian word, if one of Zbor's possible meanings appears in that translation, Zbor picks it and marks the word **confirmed in context**.

**Important:** AroTranslate was trained on a bigger version of the same corpus, so it has probably **already seen many of our test sentences**. The cell below counts how many of its translations are identical to the reference. Its scores are therefore too optimistic. The folk tales are a fairer test, but they may also be in its training data. A truly fair test needs new sentences from a speaker (section 15).

In [16]:
USE_AROTRANSLATE = True
MT_OK = False
if USE_AROTRANSLATE:
    try:
        !pip -q install ctranslate2 sentencepiece sacrebleu
        import ctranslate2, sentencepiece as spm, sacrebleu
        from huggingface_hub import snapshot_download
        mdir = snapshot_download('aronlp/NLLB-rup-ron-eng-ct2')
        _mt = ctranslate2.Translator(mdir, device='cpu', compute_type='int8')
        _sp = spm.SentencePieceProcessor(model_file=os.path.join(mdir, 'sentencepiece.bpe.model'))
        MT_OK = True
        print('AroTranslate loaded.')
    except Exception as e:
        print('AroTranslate is not available, so this section is skipped:', type(e).__name__, str(e)[:200])

def translate(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        chunk = list(texts[i:i + bs])
        toks = [['rup_Latn'] + _sp.encode(t, out_type=str) + ['</s>'] for t in chunk]
        res = _mt.translate_batch(toks, target_prefix=[['ron_Latn']] * len(chunk), beam_size=4, max_decoding_length=200)
        out += [_sp.decode([t for t in r.hypotheses[0] if t != 'ron_Latn']) for r in res]
    return out

def with_context(a_toks, mt_text):
    """For each Aromanian word: (meaning, confirmed?) using the sentence translation as context."""
    stems = Counter(strip(w)[:5] for w in tok(mt_text))
    out = []
    for w in a_toks:
        k, how = lookup(w)
        if k is None:
            out.append((None, False))
            continue
        cands = [r for r, _ in GLOSS[k]]
        conf = next((r for r in cands if stems[strip(r)[:5]] > 0), None)
        if conf:
            stems[strip(conf)[:5]] -= 1
        out.append((conf or cands[0], conf is not None))
    return out

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<?, ?it/s]

AroTranslate loaded.


In [17]:
if MT_OK:
    MT_SETS = {'corpus sentences (400)': TEST['sentences'].sample(400, random_state=0).copy(),
               'folk tales': TEST['folk tales'].copy()}
    chrf_rows, word_rows = [], []
    for name, s in MT_SETS.items():
        s['mt'] = translate(s.aro.tolist())
        refs = [s.ro.tolist()]
        outputs = {'copy the text': s.aro.tolist(),
                   'looks like Romanian (word by word)': [' '.join(looks_like(w)[0] for w in a) for a in s.aro_t],
                   'Zbor (word by word)': [' '.join(zbor_guess(w) for w in a) for a in s.aro_t],
                   'AroTranslate (whole sentence)': s.mt.tolist()}
        chrf_rows.append({'test set': name, **{k: round(sacrebleu.corpus_chrf(v, refs).score, 1) for k, v in outputs.items()},
                          'AroTranslate identical to reference': f"{(s.mt.str.strip() == s.ro.str.strip()).mean():.0%}"})
        t = Counter()
        for a_toks, r_toks, m in zip(s.aro_t, s.ro_t, s.mt):
            ref_alone, ref_ctx = Counter(strip(w)[:5] for w in r_toks), Counter(strip(w)[:5] for w in r_toks)
            for w, (g, conf) in zip(a_toks, with_context(a_toks, m)):
                g0 = zbor_guess(w)
                h0 = ref_alone[strip(g0)[:5]] > 0
                if h0: ref_alone[strip(g0)[:5]] -= 1
                h1 = g is not None and ref_ctx[strip(g)[:5]] > 0
                if h1: ref_ctx[strip(g)[:5]] -= 1
                t['n'] += 1; t['alone'] += h0; t['ctx'] += h1
                if conf: t['conf'] += 1; t['conf_hit'] += h1
        word_rows.append({'test set': name, 'Zbor alone: right': f"{t['alone'] / t['n']:.1%}",
                          'Zbor + context: right': f"{t['ctx'] / t['n']:.1%}",
                          'words confirmed in context': f"{t['conf'] / t['n']:.0%}",
                          'confirmed words that are right': f"{t['conf_hit'] / max(t['conf'], 1):.0%}"})
    print('chrF (0-100, higher = closer to the reference translation):')
    display(pd.DataFrame(chrf_rows).set_index('test set'))
    print('\nWord meanings (lenient, clipped):')
    display(pd.DataFrame(word_rows).set_index('test set'))

chrF (0-100, higher = closer to the reference translation):


,copy the text,looks like Romanian (word by word),Zbor (word by word),AroTranslate (whole sentence),AroTranslate identical to reference
test set,,,,,
corpus sentences (400),22.1,24.2,41.3,67.8,11%
folk tales,20.4,21.8,26.5,39.4,4%



Word meanings (lenient, clipped):


,Zbor alone: right,Zbor + context: right,words confirmed in context,confirmed words that are right
test set,,,,
corpus sentences (400),55.9%,65.7%,80%,80%
folk tales,41.5%,46.1%,72%,58%


**What we see:** AroTranslate's whole-sentence translations are far closer to the reference than any word-by-word method. For the corpus sentences this is partly because it has seen them before. Used as context, it lets Zbor **confirm the meaning of most words** and pick the right one among its candidates. Confirmed words are shown in green as "confirmed in context".

## 13. Side by side: what the reader sees

Folk-tale test sentences the model never saw, with the correct Romanian translation underneath. With AroTranslate available, Zbor also shows the sentence translation, and words confirmed by it are green with a ✓. Hover over a word to see its score.

In [18]:
BG = {'green': '#c8f7c5', 'yellow': '#fff3b0', 'red': '#ffc9c9'}

def explain(text, mt=None):
    pieces = list(re.finditer(r'[^\W\d_]+|[^\w\s]+|\d+|\s+', unicodedata.normalize('NFC', text).translate(CEDILLA)))
    words = [m.group(0).lower() for m in pieces if WORD.fullmatch(m.group(0))]
    ctx = iter(with_context(words, mt)) if mt else None
    out = [f'<p><b>AroTranslate:</b> {mt}</p>'] if mt else []
    for m in pieces:
        piece = m.group(0)
        if not WORD.fullmatch(piece):
            out.append(piece if not piece.isspace() else ' ')
            continue
        w = piece.lower()
        k, how = lookup(w)
        cg, conf = next(ctx) if ctx else (None, False)
        if conf:
            g, c, title = cg + ' ✓', 'green', 'confirmat în context de traducerea propoziției'
        elif k and how != 'similar':
            g, s = GLOSS[k][0]
            c, title = colour(s), f'scor {s:.2f}' + (' (altă ortografie)' if how == 'spelling' else '')
        elif k:
            g, c, title = f'≈{GLOSS[k][0][0]}', 'red', f'cuvânt necunoscut; seamănă cu „{k}”'
        elif piece[0].isupper():
            g, c, title = '(nume?)', 'red', 'cuvânt necunoscut cu majusculă: probabil un nume'
        else:
            g, c, title = f'≈{looks_like(w)[0]}?', 'red', 'cuvânt necunoscut: ghicit după formă'
        warn = ''
        if (k or w) in TRAP:
            lk_, real = TRAP[k or w]
            warn, title = ' ⚠', f'Capcană: arată ca „{lk_}”, dar înseamnă „{real}”'
        out.append(f'<span title="{title}" style="display:inline-block;margin:2px;padding:3px 6px;border-radius:6px;'
                   f'background:{BG[c]};color:#111;text-align:center;line-height:1.2"><b>{piece}{warn}</b><br>'
                   f'<small>{g}</small></span>')
    return '<div style="font-size:1.05em">' + ''.join(out) + '</div>'

for _, r in TEST['folk tales'].sample(3, random_state=7).iterrows():
    mt = translate([r.aro])[0] if MT_OK else None
    display(HTML(explain(r.aro, mt) + f'<p><i>Correct Romanian:</i> {r.ro}</p><hr>'))

## 14. Real-life test: Aromanian Wikipedia

The corpus was cleaned and converted to one standard spelling. Real Aromanian text online is messier. We use random article intros from Aromanian Wikipedia (roa-rup.wikipedia.org, CC BY-SA) and check how many words Zbor can explain, and how.

In [19]:
API = 'https://roa-rup.wikipedia.org/w/api.php'
HEADERS = {'User-Agent': 'ZborStudentHackathon/1.0 (educational project)'}
wiki = []
if os.path.exists('wiki_sample.csv'):            # reuse the saved sample so results stay the same
    wiki = pd.read_csv('wiki_sample.csv').values.tolist()
for _ in range(0 if wiki else 5):
    params = dict(action='query', generator='random', grnnamespace=0, grnlimit=20, prop='extracts',
                  exintro=1, explaintext=1, exlimit=20, format='json')
    pages = requests.get(API, params=params, headers=HEADERS, timeout=30).json().get('query', {}).get('pages', {})
    wiki += [(p['title'], p.get('extract', '')) for p in pages.values() if len(p.get('extract', '')) > 80]
wiki = pd.DataFrame(wiki, columns=['title', 'text']).drop_duplicates('title')
wiki.to_csv('wiki_sample.csv', index=False)
print(len(wiki), 'articles')

wtoks = [w for t in wiki.text for w in tok(t)]
how_found = Counter(lookup(w)[1] or 'unknown' for w in wtoks)
print(f'{len(wtoks):,} words:', {k: f'{v / len(wtoks):.1%}' for k, v in how_found.most_common()})
print('\nMost common words Zbor does NOT know at all:')
print(Counter(w for w in wtoks if lookup(w)[0] is None).most_common(25))
display(HTML(explain(wiki.text.iloc[0][:600])))

72 articles


6,194 words: {'known': '66.5%', 'similar': '18.6%', 'unknown': '13.0%', 'spelling': '1.9%'}

Most common words Zbor does NOT know at all:
[('dicționarlu', 24), ('tiberius', 24), ('scopia', 20), ('etc', 16), ('rm', 13), ('educatsii', 12), ('developari', 9), ('ministeriatlu', 9), ('mihailova', 8), ('initsiativã', 8), ('natsional', 7), ('premii', 7), ('partitsipari', 7), ('ue', 6), ('membrã', 5), ('r', 5), ('integra', 5), ('sotsietati', 5), ('tsivilã', 5), ('ohãrda', 5), ('redactor', 4), ('facultatea', 4), ('wikipedia', 4), ('în', 4), ('colaborator', 4)]


In [20]:
# 50 random words in context for our own hand check, and later the Aromanian speaker's check.
rng = random.Random(0)
sample = []
for title, text in wiki.sample(min(15, len(wiki)), random_state=0).values:
    words = tok(text)
    for i in rng.sample(range(len(words)), min(4, len(words))):
        w = words[i]
        k, how = lookup(w)
        sample.append({'word': w, 'context': ' '.join(words[max(0, i - 4): i + 5]),
                       'zbor says': GLOSS[k][0][0] if k else f'≈{looks_like(w)[0]}?',
                       'found as': how or 'unknown', 'correct? (fill in)': ''})
check = pd.DataFrame(sample[:50])
check.to_csv('hand_check.csv', index=False)
display(check.head(15))

,word,context,zbor says,found as,correct? (fill in)
0,tu,regiunea veliko tãrnovo vãryãrie tu anlu cãsãbãlu avea bãnãtori,în,known,
1,svishtov,trã anlu fu cã svishtov ari bãnãtori,≈ariston?,unknown,
2,anlu,veliko tãrnovo vãryãrie tu anlu cãsãbãlu avea bãnãtori sh,al,known,
3,vãryãreashti,svishtov vãryãreashti свищов easti unu cãsãbã,bulgărește,known,
4,latini,marea figuri a muzicãljei latini dit lumi numatã adesea,latine,known,
5,t,la tortura hips don t lie loca waka waka,o,known,
6,tortura,ca whenever wherever la tortura hips don t lie,torturi,similar,
7,agiumsi,nica di njicã ea agiumsi la un suctses mondial,a,known,
8,biseritsi,anãltsari di acaretsi casi biseritsi pãlãts puntsã etc tu,se,known,
9,dicționarlu,ari textu di tu dicționarlu alu tiberius cunia ți,≈picioare?,unknown,


**Where it fails:**
- **Real text is harder than the corpus.** On Wikipedia, many words are unknown or only guessed from a similar word. Most are names (*Svishtov*, *Tiberius*), modern words the Bible-heavy corpus never had (*educatsii*), or text in other languages inside the article.
- **One meaning per word.** Zbor can't use the surrounding words to choose between meanings.
- **Inflection.** The right word in a different form (*latini* → "latine") counts as wrong in the exact score.
- **Borrowed meanings can be wrong.** A similar-looking known word is sometimes a different word, which is why those are always shown in red.
- **Some traps are probably false alarms**, e.g. when the "looks like" Romanian word is one no reader would really think of. A speaker has to check the list.

### Everyday phrases from public phrasebooks

The corpus is mostly Bible text, so we also built a small **everyday** test set: 55 greetings, questions and common words (*Bunã dzua*, *Cumu eshti?*, *Haristo*, *Ti multsã-anji!*, *aoa*, *babã*...) collected from public sources with their Romanian meaning: English and Romanian Wikipedia, Wiktionary, g4media, Omniglot and the Aromanian Wikipedia dictionary (`phrasebook_test.csv`, with a source for each line). Only items a source confirmed are used; unconfirmed ones were left out.

Two cautions: the reference translations are idiomatic (*Di cãtsi anji esht?* is literally "de câți ani ești?" but the reference says "câți ani ai?"), so word-by-word methods are judged strictly; and 55 items is a small sample.

In [21]:
if os.path.exists('phrasebook_test.csv'):
    pb = prepare(pd.read_csv('phrasebook_test.csv').astype({'aro': str, 'ro': str}))
    parts = {'all 55': pb, 'phrases': pb[pb.kind == 'phrase'], 'single words': pb[pb.kind == 'word']}
    print('Lenient word matches on the phrasebook test:')
    display(pd.DataFrame({name: {p: evaluate(part, f, aro_counts)['lenient'] for p, part in parts.items()}
                          for name, f in methods.items()}).T)
    known = [w for t in pb.aro_t for w in t]
    print(f"{sum(lookup(w)[1] == 'known' for w in known) / len(known):.0%} of the phrasebook's words are in Zbor's dictionary")
    side = pb[['aro', 'ro']].copy()
    side['Zbor (word by word)'] = [' '.join(zbor_guess(w) for w in t) for t in pb.aro_t]
    if MT_OK:
        side['AroTranslate'] = translate(pb.aro.tolist())
        ph = side[pb.kind == 'phrase']
        print('chrF on the phrases:', {'Zbor': round(sacrebleu.corpus_chrf(ph['Zbor (word by word)'].tolist(), [ph.ro.tolist()]).score, 1),
                                       'AroTranslate': round(sacrebleu.corpus_chrf(ph['AroTranslate'].tolist(), [ph.ro.tolist()]).score, 1)})
    display(side.rename(columns={'aro': 'Aromanian', 'ro': 'Correct Romanian'}))
else:
    print('phrasebook_test.csv not found.')

Lenient word matches on the phrasebook test:


,all 55,phrases,single words
"always ""și""",0.0,0.0,0.0
copy word,16.5,18.8,12.1
looks like Romanian,22.7,26.6,15.2
Zbor (ours),60.8,56.2,69.7


88% of the phrasebook's words are in Zbor's dictionary


chrF on the phrases: {'Zbor': 42.3, 'AroTranslate': 57.5}


,Aromanian,Correct Romanian,Zbor (word by word),AroTranslate
0,Bunã dzua,Bună ziua,bună ziua,Ziua bună
1,Cum ti chiamã,Cum te cheamă,cum pentru cheamă,Cum te cheamă
2,Di cãtsi anji esht,Câți ani ai,de dans ani ești,De câți ani ești
3,Cumu hits,Ce mai faceți,cum sunteți,Cum sunteți
4,Cumu eshti,Ce mai faci,cum ești,Cum ești
5,Tsi fats,Ce faci,ce faci,Ce faci?
6,S'nã videmu cu ghine,La revedere,să ne nu cu bine,Să ne vedem cu bine
7,S'nã avdzãmu ghiniatsa,Numai bine,să ne am bine,Să ne auzim binele
8,Vã plãcãrsescu,Vă rog,vă rog,Vă rog
9,S'mi hãrãdzesht,Iartă-mă,să mă hărăzit,Să mă fericiți


## 15. Real-speaker check

**a) Sentences translated by a real speaker.** We prepared 25 everyday Romanian sentences for an Aromanian speaker to translate the way they would naturally write. If we get them back (`speaker_sentences.csv`, columns `id, ro, aro`), this cell tests Zbor on them.

In [22]:
if os.path.exists('speaker_sentences.csv'):
    sp_ = prepare(pd.read_csv('speaker_sentences.csv').dropna().astype(str))
    display(pd.DataFrame({name: evaluate(sp_, f, aro_counts) for name, f in methods.items()}).T)
    for _, r in sp_.head(5).iterrows():
        display(HTML(explain(r.aro) + f'<p><i>Original Romanian:</i> {r.ro}</p><hr>'))
else:
    print('No speaker_sentences.csv yet.')

No speaker_sentences.csv yet.


**b) Verdicts on our trap list and glosses.** We contacted Aromanian community groups and asked a speaker to check our trap list (`traps.csv`) and 50 glossed words (`hand_check.csv`). **Nobody had replied before the deadline**, so none of our traps or glosses have been confirmed by a native speaker yet. This is the most important next step. Until then, the trap list should be treated as "possible traps".

In [23]:
SPEAKER_CHECK = {
    # 'zbor': 'correct trap (cuvânt, not zbor)',
}
print(f'{len(SPEAKER_CHECK)} items checked by an Aromanian speaker')

0 items checked by an Aromanian speaker


## 16. A second language: Romani (Lav)

Romanians mix up two languages whose names both start with "Rom-": **Aromanian**, a Romance language very close to Romanian, and **Romani**, the language of the Roma, an Indo-Aryan language that is not related to Romanian at all. Romanian slang has borrowed a few Romani words (*mișto*, *șucar*, *gagiu*, *lovele*), so Romanians often think they know some Romani too, and people online sometimes call Romani "the Aromanian language". So we built a second helper, **Lav** (*lav* means "word" in Romani, just like *zbor* in Aromanian), with exactly the same method.

**Data:** the Romanian–Romani Bible from OPUS (`bible-uedin`, Christodoulopoulos & Steedman 2015, CC0): about 7,900 verse pairs in Kalderash Romani, the Romani dialect most spoken in Romania. That is 8 times less data than for Aromanian, and it is all Bible text. We split it 90/5/5 like the main corpus. To get more data, we added a second Romani Bible: the **Chergash New Testament** (a Vlax Romani dialect, like the Romani of Romania; ebible.org, © 2018 Pioneers, CC BY-SA 4.0), paired verse by verse with the modern Romanian **Biblia Liberă** (ebible.org, public domain). It is used only for training, and every verse that is in our dev or test set is left out of it. The Romanian side uses the old spelling (*sînt*, *cîine*), so we convert it to today's spelling (*sunt*, *câine*).

**Spelling:** Romani has no single spelling. The Bible writes *so keres*, *mishto*, *khere*; people on social media write *socares*, *misto*, *care* (Romanian-style). So for unknown words Lav compares a simplified "skeleton" of the spelling (*ch, kh, c, k* → *k*; *sh, ș, š* → *s*; *zh, j, dž* → *z*; no *h*), which lets *misto* find *mishto* and *cai* find *kai*.

In [24]:
ROM_URL = 'https://object.pouta.csc.fi/OPUS-bible-uedin/v1/moses/ro-rom.txt.zip'
if os.path.exists('romani_data/bible-uedin.ro-rom.rom'):
    read_rom = lambda ext: open(f'romani_data/bible-uedin.ro-rom.{ext}', encoding='utf-8').read().split('\n')
else:
    zr = zipfile.ZipFile(io.BytesIO(requests.get(ROM_URL, timeout=120).content))
    read_rom = lambda ext: zr.read(f'bible-uedin.ro-rom.{ext}').decode('utf-8').split('\n')

OLD_RO = {'sînt': 'sunt', 'sîntem': 'suntem', 'sînteți': 'sunteți'}
def modern(w):
    if w in OLD_RO:
        return OLD_RO[w]
    return re.sub(r'^sânt', 'sunt', re.sub(r'(?<=\w)î(?=\w)', 'â', w))

verse_ids = [re.search(r';b\.([^.]+)\.(\d+)\.(\d+)', l).groups() for l in read_rom('xml') if 'xtargets' in l]
rdf = pd.DataFrame({'rom': read_rom('rom')[:len(verse_ids)], 'ro': read_rom('ro')[:len(verse_ids)],
                    'ref': [f'{b} {c}:{v}' for b, c, v in verse_ids]})   # e.g. "MAT 5:11"
rdf = rdf[(rdf.rom.str.strip() != '') & (rdf.ro.str.strip() != '')].copy()
rdf['rom_t'], rdf['ro_t'] = rdf.rom.map(tok), rdf.ro.map(lambda s: [modern(w) for w in tok(s)])
rdf = rdf[rdf.rom_t.str.len().between(1, 60) & rdf.ro_t.str.len().between(1, 60)].drop_duplicates('rom')
rdf = rdf.sample(frac=1, random_state=42).reset_index(drop=True)
nr = len(rdf)
r_test, r_dev, r_train = rdf[:int(.05 * nr)], rdf[int(.05 * nr):int(.10 * nr)], rdf[int(.10 * nr):]
print(f'{nr:,} Romani–Romanian verse pairs: {len(r_train):,} train | {len(r_dev)} dev | {len(r_test)} test')
print('Example:', rdf.rom.iloc[0][:100], '→', rdf.ro.iloc[0][:100])

# Extra data: the Chergash (Vlax Romani) New Testament, paired verse by verse with the modern Romanian
# "Biblia Liberă". Only for training: every verse in the dev or test set is removed, so no test verse leaks in.
def read_vpl(eb_id):
    path = f'romani_data/ebible/{eb_id}/{eb_id}_vpl.txt'
    if os.path.exists(path):
        lines = open(path, encoding='utf-8').read().split('\n')
    else:
        z = zipfile.ZipFile(io.BytesIO(requests.get(f'https://ebible.org/Scriptures/{eb_id}_vpl.zip', timeout=120).content))
        lines = z.read(f'{eb_id}_vpl.txt').decode('utf-8').split('\n')
    return {m.group(1): m.group(2) for m in (re.match(r'(\S+ \d+:\d+) (.*)', l.strip()) for l in lines) if m}
chergash, biblia_libera = read_vpl('rmyChergash'), read_vpl('ronbl')
held_out = set(r_test.ref) | set(r_dev.ref)
r_extra = pd.DataFrame([{'rom': chergash[k], 'ro': biblia_libera[k], 'ref': k}
                        for k in chergash if k in biblia_libera and k not in held_out])
r_extra['rom_t'], r_extra['ro_t'] = r_extra.rom.map(tok), r_extra.ro.map(tok)
r_extra = r_extra[r_extra.rom_t.str.len().between(1, 60) & r_extra.ro_t.str.len().between(1, 60)]
print(f'Extra training pairs from the Chergash NT: {len(r_extra):,} '
      f'({len(held_out & set(chergash))} dev/test verses left out)')
print('Example:', r_extra.rom.iloc[100][:90], '→', r_extra.ro.iloc[100][:90])

# Same IBM Model 1 as Zbor, Romani -> Romanian: 10 rounds and the position prior (both chosen on dev, see the table below).
def train_lav(tr, iters=10, diag=4.0):
    ac, rc = Counter(w for s in tr.rom_t for w in s), Counter(w for s in tr.ro_t for w in s)
    a_itos, r_itos_ = list(ac), list(rc)
    p = ibm1(tr.rom_t, tr.ro_t, {w: i for i, w in enumerate(a_itos)}, {w: i for i, w in enumerate(r_itos_)}, iters, diag=diag)
    top = p.sort_values('p', ascending=False).groupby('src').head(5)
    G = {}
    for a, r, s in zip(top.src, top.tgt, top.p):
        G.setdefault(a_itos[a], []).append((r_itos_[r], float(s)))
    return G, ac, rc
r_train_all = pd.concat([r_train, r_extra], ignore_index=True)
R_GLOSS, rom_counts, r_ro_counts = train_lav(r_train_all)

SKEL = [('đ', 'z'), ('dzh', 'z'), ('dž', 'z'), ('dj', 'z'), ('zh', 'z'), ('ž', 'z'), ('j', 'z'), ('dz', 'z'),
        ('ch', 'k'), ('kh', 'k'), ('č', 'k'), ('ć', 'k'), ('c', 'k'), ('q', 'k'),
        ('sh', 's'), ('š', 's'), ('ș', 's'), ('ś', 's'), ('ț', 'ts'),
        ('ph', 'p'), ('th', 't'), ('x', 'h'), ('y', 'i'), ('w', 'v'), ('ă', 'a'), ('â', 'i'), ('î', 'i')]
def skeleton(w):
    w = w.lower()
    for a, b in SKEL:
        w = w.replace(a, b)
    w = strip(w).replace('h', '')
    return re.sub(r'(.)\1+', r'\1', w)

BY_SKEL = {}
for w, _ in rom_counts.most_common():
    BY_SKEL.setdefault(skeleton(w), w)   # most frequent word for each skeleton
SKEL_KEYS = list(BY_SKEL)

R_RO_VOCAB = [w for w, c in r_ro_counts.items() if c >= 2]
R_RO_KEYS = [strip(w) for w in R_RO_VOCAB]
@lru_cache(maxsize=None)
def r_looks_like(w):
    return R_RO_VOCAB[process.extractOne(strip(w), R_RO_KEYS, scorer=Levenshtein.normalized_similarity)[2]]

@lru_cache(maxsize=None)
def r_lookup(w, fold=True):
    if w in R_GLOSS:
        return w, 'known'
    if not fold:
        return None, None
    s = skeleton(w)
    if s in BY_SKEL:
        return BY_SKEL[s], 'spelling'
    m = process.extractOne(s, SKEL_KEYS, scorer=Levenshtein.normalized_similarity, score_cutoff=0.75)
    return (BY_SKEL[m[0]], 'similar') if m else (None, None)

def r_guess_with(fold):
    def g(w):
        k, _ = r_lookup(w, fold)
        return R_GLOSS[k][0][0] if k else r_looks_like(w)
    return g
lav_guess = r_guess_with(True)
print(f'Lav dictionary: {len(R_GLOSS):,} Romani words')
for w in ['so', 'keres', 'mishto', 'misto', 'phral', 'devla', 'bari', 'cai', 'lovo']:
    k, how = r_lookup(w)
    print(f'{w:8} -> {k} ({how}):', [r for r, _ in R_GLOSS[k][:3]] if k else '-')

7,900 Romani–Romanian verse pairs: 7,110 train | 395 dev | 395 test
Example: Akana mothav tumenge te len xaben; ke trobul te xan te traiin: ai chi iek firo anda tumare bal chi x → Deaceea vă rog să mîncaţi, căci lucrul acesta este pentru scăparea voastră; şi nu vi se va pierde ni
Extra training pairs from the Chergash NT: 7,157 (790 dev/test verses left out)
Example: Čače phenav tuće: či inkljeja okotar dok či poćines sa so san udžile dži ko poslednjo lovo → Adevărat îți spun că nicidecum nu vei ieși de acolo până nu vei plăti și ultimul bănuț.


Lav dictionary: 15,382 Romani words
so       -> so (known): ['ce', 'să', 'de']
keres    -> keres (known): ['faci', 'judeci', 'ți']
mishto   -> mishto (known): ['bine', 'cu', 'este']
misto    -> mishto (spelling): ['bine', 'cu', 'este']
phral    -> phral (known): ['frații', 'fratele', 'fraților']
devla    -> devla (known): ['doamne', 'dumnezeule', 'răspuns']
bari     -> bari (known): ['mare', 'o', 'de']
cai      -> kai (spelling): ['care', 'pe', 'cari']
lovo     -> slovo (similar): ['literă', 'unui', 'al']


**Results.** The same baselines and the same clipped scoring as for Aromanian. Besides the Bible test verses, we test on **everyday Romani**: 38 greetings, questions and words written the way people really type them online (*Socares?*, *Cai san?*, *mishto*), with their Romanian meaning, from Wiktionary, dexonline, Dilema Veche, a Stockholm University Romani worksheet and an online phrase list (`romani_phrasebook_test.csv`, one source per line). Insults and anything not family-friendly were left out. A third small set holds 12 Romani words and phrases most Romanians already know from popular songs and manele (*Tu san shukar*, *Chaiorie*, *Kamav tut*, *Bahtale Romale*, *Gelem gelem*; `romani_songs_test.csv`, with the song and a source for each meaning). We use only titles and single well-known words, never song lyrics.

In [25]:
TOP_R = r_ro_counts.most_common(1)[0][0]
r_methods = {f'always "{TOP_R}"': lambda w: TOP_R, 'copy word': lambda w: w,
             'looks like Romanian': r_looks_like, 'Lav without spelling skeleton': r_guess_with(False),
             'Lav (ours)': lav_guess}
r_sets = {'Bible verses (test)': r_test}
if os.path.exists('romani_phrasebook_test.csv'):
    rpb = pd.read_csv('romani_phrasebook_test.csv')
    rpb['rom_t'], rpb['ro_t'] = rpb.rom.map(tok), rpb.ro.map(tok)
    r_sets.update({'everyday (all 38)': rpb, 'everyday phrases': rpb[rpb.kind == 'phrase'],
                   'everyday words': rpb[rpb.kind == 'word']})
if os.path.exists('romani_songs_test.csv'):   # Romani that most Romanians know from popular songs
    rsong = pd.read_csv('romani_songs_test.csv')
    rsong['rom_t'], rsong['ro_t'] = rsong.rom.map(tok), rsong.ro.map(tok)
    r_sets['from popular songs (12)'] = rsong
rom_final = pd.DataFrame({s: {m: evaluate(part, f, rom_counts, src_col='rom_t')['lenient'] for m, f in r_methods.items()}
                          for s, part in r_sets.items()})
print('Lenient word matches (%):')
display(rom_final)
# Experiments: which training data and how many rounds? DEV decides; the other columns are only reported.
def lav_guess_from(G, ac, rc):
    by = {}
    for w, _ in ac.most_common():
        by.setdefault(skeleton(w), w)
    keys, RV = list(by), [w for w, c in rc.items() if c >= 2]
    RK = [strip(w) for w in RV]
    @lru_cache(maxsize=None)
    def guess(w):
        k = w if w in G else by.get(skeleton(w))
        if k is None:
            m = process.extractOne(skeleton(w), keys, scorer=Levenshtein.normalized_similarity, score_cutoff=0.75)
            k = by[m[0]] if m else None
        return G[k][0][0] if k else RV[process.extractOne(strip(w), RK, scorer=Levenshtein.normalized_similarity)[2]]
    return guess
twins = r_train[r_train.ref.isin(biblia_libera)].copy()   # the same Kalderash verses, with the modern Romanian text
twins['ro'] = twins.ref.map(biblia_libera)
twins['ro_t'] = twins.ro.map(tok)
lav_exp = []
for name, tr, it, dg, is_final in [('Kalderash Bible only, 6 rounds', r_train, 6, 0.0, False),
                                ('+ Chergash NT, 6 rounds', r_train_all, 6, 0.0, False),
                                ('+ Chergash NT, 10 rounds', r_train_all, 10, 0.0, False),
                                ('+ position prior (final)', r_train_all, 10, 4.0, True),
                                ('+ modern-Romanian twins of the Kalderash verses', pd.concat([r_train_all, twins]), 10, 4.0, False)]:
    G_, ac_, rc_ = (R_GLOSS, rom_counts, r_ro_counts) if is_final else train_lav(tr, it, dg)
    f_ = lav_guess_from(G_, ac_, rc_)
    lav_exp.append({'version': name, 'training pairs': len(tr), 'Romani words': len(G_),
                    'DEV verses': evaluate(r_dev, f_, ac_, 'rom_t')['lenient'],
                    **{s: evaluate(p_, f_, ac_, 'rom_t')['lenient'] for s, p_ in r_sets.items() if s != 'Bible verses (test)'}})
lav_exp = pd.DataFrame(lav_exp).set_index('version')
print('Lenient word matches (%). We pick the version with the best DEV score:')
display(lav_exp)
tw, fin = lav_exp['DEV verses'].iloc[-1], lav_exp.loc['+ position prior (final)', 'DEV verses']
print(f'Modern twins on DEV: {tw} vs {fin} for the final version, so we {"did not use" if tw <= fin else "should use"} them.')

# Confidence colours for Lav, tuned on the Romani dev verses the same way as for Zbor:
# green = the lowest score at which words are right at least 80% of the time, yellow = at least 60%.
def r_scored(split):
    rows = []
    for a_toks, r_toks in zip(split.rom_t, split.ro_t):
        stems = Counter(strip(w)[:5] for w in r_toks)
        for w in a_toks:
            k, how = r_lookup(w)
            if k and how != 'similar':
                g, s = R_GLOSS[k][0]
                rows.append((s, stems[strip(g)[:5]] > 0))
    return pd.DataFrame(rows, columns=['score', 'right'])
r_cal_dev = r_scored(r_dev)
def lowest_threshold(cal, target):
    best = 0.99
    for th in np.round(np.arange(0.99, 0.01, -0.01), 2):
        sel = cal[cal.score >= th]
        if len(sel) >= 30 and sel.right.mean() >= target:
            best = th
    return float(best)
R_GREEN, R_YELLOW = lowest_threshold(r_cal_dev, 0.80), lowest_threshold(r_cal_dev, 0.60)
r_cal_test = r_scored(r_test)
r_cal_test['colour'] = np.where(r_cal_test.score >= R_GREEN, 'green', np.where(r_cal_test.score >= R_YELLOW, 'yellow', 'red'))
r_calib = r_cal_test.groupby('colour').right.agg(['mean', 'size']).rename(columns={'mean': '% right on TEST', 'size': 'words'})
r_calib['% right on TEST'] = (100 * r_calib['% right on TEST']).round(1)
print(f'Lav colours: green if score >= {R_GREEN}, yellow if >= {R_YELLOW}')
display(r_calib)
if os.path.exists('romani_phrasebook_test.csv'):
    side_r = rpb[['rom', 'ro']].copy()
    side_r['Lav (word by word)'] = [' '.join(lav_guess(w) for w in t) for t in rpb.rom_t]
    display(side_r.rename(columns={'rom': 'Romani (as typed online)', 'ro': 'Correct Romanian'}).head(17))
if 'rsong' in globals():
    side_s = rsong[['rom', 'ro', 'heard_in']].copy()
    side_s['Lav (word by word)'] = [' '.join(lav_guess(w) for w in t) for t in rsong.rom_t]
    display(side_s.rename(columns={'rom': 'Romani from songs', 'ro': 'Correct Romanian', 'heard_in': 'Where Romanians hear it'}))

# An idea we tried and dropped: very common Romanian words (de, și, să) soak up probability, so *shukar*
# comes out as "de". We tried skipping a top meaning that is one of the 20 most common Romanian words.
FUNC20 = {w for w, _ in r_ro_counts.most_common(20)}
def lav_skip_common(w):
    k, _ = r_lookup(w)
    if not k:
        return r_looks_like(w)
    opts = R_GLOSS[k]
    if len(w) >= 4 and opts[0][0] in FUNC20:
        for r, s in opts[1:]:
            if r not in FUNC20 and s >= 0.08:
                return r
    return opts[0][0]
print('Dev verses, lenient:', {'Lav': evaluate(r_dev, lav_guess, rom_counts, 'rom_t')['lenient'],
                               'Lav, skipping common words': evaluate(r_dev, lav_skip_common, rom_counts, 'rom_t')['lenient']})
print('It makes the dev score worse, so we did not use it (it would only help on the test sets, which we may not tune on).')

Lenient word matches (%):


,Bible verses (test),everyday (all 38),everyday phrases,everyday words,from popular songs (12)
"always ""și""",4.0,1.5,2.2,0.0,0.0
copy word,3.9,4.5,6.5,0.0,4.8
looks like Romanian,6.7,3.0,4.3,0.0,4.8
Lav without spelling skeleton,38.9,32.8,32.6,33.3,42.9
Lav (ours),39.1,43.3,41.3,47.6,42.9


Lenient word matches (%). We pick the version with the best DEV score:


,training pairs,Romani words,DEV verses,everyday (all 38),everyday phrases,everyday words,from popular songs (12)
version,,,,,,,
"Kalderash Bible only, 6 rounds",7110,6741,39.8,35.8,32.6,42.9,33.3
"+ Chergash NT, 6 rounds",14267,15382,39.9,43.3,41.3,47.6,38.1
"+ Chergash NT, 10 rounds",14267,15382,40.0,43.3,41.3,47.6,38.1
+ position prior (final),14267,15382,40.4,43.3,41.3,47.6,42.9
+ modern-Romanian twins of the Kalderash verses,21369,15382,39.8,44.8,43.5,47.6,42.9


Modern twins on DEV: 39.8 vs 40.4 for the final version, so we did not use them.


Lav colours: green if score >= 0.72, yellow if >= 0.35


,% right on TEST,words
colour,,
green,80.0,950
red,37.0,3810
yellow,55.9,4464


,Romani (as typed online),Correct Romanian,Lav (word by word)
0,Dobroi tuca,bună ziua,dori ți
1,Socares?,Ce faci?,înveți
2,Socares prala?,"Ce faci, frate?",înveți fraților
3,So halean?,Ce ai mâncat?,ce asupra
4,Mai bes hari,Mai stai puțin,mai șezi dacă
5,Haida jas manta ando foro?,Mergi cu mine în oraș?,vino ne mine în cetate
6,Camis te jas manta?,Vrei să mergi cu mine?,vrei să ne mine
7,Hai vi tu avri,Hai și tu afară,și și tu afară
8,Haida care,Hai acasă,vino ea
9,Sar busos?,Cum te cheamă?,cum numele


,Romani from songs,Correct Romanian,Where Romanians hear it,Lav (word by word)
0,Chaiorie,fetițo,"song title: Ionuț Cercel, ""Chaiorie""; Sofi Marinova, ""Chaiorie Shukarie""",călătorie
1,Chaiorie shukarie,fetiță frumoasă,"song title: Sofi Marinova, ""Chaiorie Shukarie""",călătorie podoaba
2,Tu san shukar,tu ești frumoasă,common in manele and Roma songs,tu ești de
3,shukar,frumos,common in manele and Roma songs,de
4,Kamav tut,te iubesc,common in Roma love songs,vreau te
5,Bahtale Romale,romi norocoși,"song: Gipsy Casual, ""Romale""; Bahtalo band, ""Bahtalo Romale""",bucurie soților
6,Gelem gelem,am mers am mers,"the Romani anthem, ""Gelem, Gelem""",am am
7,Loli phabaj,măr roșu,"song title: Esma Redžepova, ""Loli phabaj""",roșie piață
8,Devla,Doamne,common in manele and Roma songs,doamne
9,mishto,bine,common in manele; became Romanian slang,bine


Dev verses, lenient: {'Lav': 40.4, 'Lav, skipping common words': 39.2}
It makes the dev score worse, so we did not use it (it would only help on the test sets, which we may not tune on).


**Which language is it?** Before explaining a text, the website has to know whether it is Aromanian, Romani or plain Romanian. We trained a small **character n-gram language detector** (naive Bayes over 1- to 4-letter pieces of words) on the three training sets. Because people write Romani with Romanian letters online, we also add a copy of the Romani training text re-spelled the Romanian way (*kh* → *c*, *sh* → *ș* or *s*, *zh* → *j*). We test on sentences it never saw, on their first 3 words only, on single words, and on both everyday phrase lists.

In [26]:
def grams(s):
    out = []
    for w in tok(s):
        w = f'_{w}_'
        for k in (1, 2, 3, 4):
            out += [w[i:i + k] for i in range(len(w) - k + 1)]
    return out

RO_STYLE = [('chi', 'ci'), ('che', 'ce'), ('ch', 'ci'), ('kh', 'k'), ('ke', 'che'), ('ki', 'chi'), ('k', 'c'),
            ('sh', 'ș'), ('zh', 'j'), ('ph', 'p'), ('th', 't'), ('x', 'h')]
def ro_style(s, diacritics=True):
    s = s.lower()
    for a, b in RO_STYLE:
        s = s.replace(a, b)
    return s if diacritics else s.replace('ș', 's')

# Only the Kalderash text: when we also added the Chergash NT here, everyday Romani typed the Romanian way was
# detected less often (89.5% -> 78.9%), because Chergash spells with č, ć, đ, unlike how people type online.
rom_tr_text = r_train.rom.tolist()
LID_TRAIN = {'aro': train.aro.tolist() + tales_train.aro.tolist(),
             'rom': rom_tr_text + [ro_style(s) for s in rom_tr_text[::2]] + [ro_style(s, False) for s in rom_tr_text[1::2]],
             'ro': train.ro.tolist()}
gcounts = {k: Counter(g for s in v for g in grams(s)) for k, v in LID_TRAIN.items()}
G_VOCAB = set()
for c in gcounts.values():
    G_VOCAB |= {g for g, _ in c.most_common(6000)}
LID = {}
for k, c in gcounts.items():
    total = sum(c[g] for g in G_VOCAB) + len(G_VOCAB) + 1
    LID[k] = {'p': {g: round(math.log((c[g] + 1) / total), 2) for g in G_VOCAB}, 'unk': round(math.log(1 / total), 2)}

def detect(s):
    gs = grams(s)
    if not gs:
        return None
    return max(LID, key=lambda k: sum(LID[k]['p'].get(g, LID[k]['unk']) for g in gs))

LID_TEST = {'aro': test.aro.tolist(), 'rom': r_test.rom.tolist(), 'ro': test.ro.tolist()}
lid_acc = lambda texts, lab: round(100 * np.mean([detect(t) == lab for t in texts]), 1)
lid_rows = {}
for lab, texts in LID_TEST.items():
    toks = [tok(t) for t in texts]
    lid_rows[{'aro': 'Aromanian', 'rom': 'Romani', 'ro': 'Romanian'}[lab]] = {
        'sentences': lid_acc(texts, lab),
        'first 3 words': lid_acc([' '.join(t[:3]) for t in toks if t], lab),
        'one word (4+ letters)': lid_acc([t[0] for t in toks if t and len(t[0]) >= 4], lab)}
lid_table = pd.DataFrame(lid_rows).T
print('Language detection accuracy (%):')
display(lid_table)
lid_everyday = {'Aromanian phrasebook (55)': lid_acc(pd.read_csv('phrasebook_test.csv').aro.tolist(), 'aro') if os.path.exists('phrasebook_test.csv') else None,
                'Romani as typed online (38)': lid_acc(rpb.rom.tolist(), 'rom') if os.path.exists('romani_phrasebook_test.csv') else None}
print(lid_everyday)

Language detection accuracy (%):


,sentences,first 3 words,one word (4+ letters)
Aromanian,96.8,95.6,88.9
Romani,100.0,100.0,98.1
Romanian,95.4,94.2,87.8


{'Aromanian phrasebook (55)': np.float64(90.9), 'Romani as typed online (38)': np.float64(89.5)}


**Common mix-ups.** Some short words are spelled the same in Aromanian and Romani (and often Romanian) but mean different things. These are the words where guessing the language wrong gives a wrong meaning, so the website shows them next to the results.

In [27]:
mix = []
for w in set(GLOSS) & set(R_GLOSS):
    ca, cr = aro_counts.get(w, 0), rom_counts.get(w, 0)
    a_m, r_m = GLOSS[w][0][0], R_GLOSS[w][0][0]
    if ca >= 20 and cr >= 20 and strip(a_m)[:4] != strip(r_m)[:4]:
        mix.append({'word': w, 'Aromanian meaning': a_m, 'Romani meaning': r_m, 'seen (aro)': ca, 'seen (rom)': cr})
mixups = pd.DataFrame(mix).sort_values('seen (rom)', ascending=False).reset_index(drop=True)
print(f'{len(mixups)} words are common in both languages but mean different things. The top 15:')
display(mixups.head(15))

43 words are common in both languages but mean different things. The top 15:


,word,Aromanian meaning,Romani meaning,seen (aro),seen (rom)
0,o,o,a,185,13566
1,e,da,și,121,10219
2,te,ce,să,166,9219
3,ai,ai,și,490,8398
4,le,le,și,45,6988
5,si,se,este,2472,4968
6,sa,să,toate,837,1764
7,numa,numele,dar,630,1502
8,na,ne,nu,469,1358
9,anda,când,din,1060,1095


## 17. Demo (Aromanian)

Set `RUN_APP = True`, paste Aromanian text, and hover over a word to see why it was flagged.

In [28]:
RUN_APP = False
if RUN_APP:
    import gradio as gr
    gr.Interface(fn=lambda text: explain(text, translate([text])[0] if MT_OK else None),
                 inputs=gr.Textbox(lines=4, label='Text în aromână', value=TEST['folk tales'].aro.iloc[0]),
                 outputs=gr.HTML(label='Zbor'),
                 title='Zbor: cuvânt cu cuvânt din aromână',
                 description='Verde = sigur · Galben = nesigur · Roșu = necunoscut sau ghicit · ⚠ = capcană (arată ca un cuvânt românesc, dar înseamnă altceva)').launch()

## 18. Export the trained models for the website

The Zbor website runs entirely in the reader's browser. It needs only what we learned here, saved as two files: `zbor_aro.js` (Zbor, loaded first) and `zbor_rom.js` (Lav, the language detector and the mix-ups). They are JavaScript files rather than `.json` so the page also works when you just double-click `index.html`. Each Romanian meaning is stored once and referred to by number, which makes the files about half as big:
- the learned dictionary (up to 3 meanings per Aromanian word, with scores);
- the colour thresholds and the trap list;
- the Romanian word list used by the "looks like" fallback;
- a few example texts and our test results.

In [29]:
import json
ex_zbor = df[df.aro_t.map(lambda t: 'zbor' in t and 5 <= len(t) <= 14)].aro.head(1).tolist()
examples = ([{'label': 'zbor in a sentence', 'aro': a} for a in ex_zbor]
            + [{'label': f'Folk tale {i + 1}', 'aro': a}
               for i, a in enumerate(TEST['folk tales'][TEST['folk tales'].aro_t.str.len().between(8, 25)].aro.head(3))]
            + [{'label': 'Aromanian Wikipedia', 'aro': wiki.text.iloc[0][:400]}])
if MT_OK:   # the website can't run AroTranslate, so we store its translation of each example
    for ex, m in zip(examples, translate([e['aro'] for e in examples])):
        ex['mt'] = m
def pack_gloss(G):
    """Store each Romanian meaning once: {word: [meaning id, score in thousandths, ...]} plus the list of meanings."""
    vocab, idx, out = [], {}, {}
    for a, opts in G.items():
        flat = []
        for r, s in [(r, s) for r, s in opts if s >= 0.05] or [opts[0]]:
            if r not in idx:
                idx[r] = len(vocab)
                vocab.append(r)
            flat += [idx[r], int(round(s * 1000))]
        out[a] = flat
    return {'vocab': vocab, 'gloss': out}

# Our sourced phrase lists. The website shows their meaning in a separate "from our phrase list" box;
# they are never used to train or tune the models.
known_phrases = []
for file, lang, col in [('phrasebook_test.csv', 'aro', 'aro'), ('romani_phrasebook_test.csv', 'rom', 'rom'), ('romani_songs_test.csv', 'rom', 'rom')]:
    if os.path.exists(file):
        for r in pd.read_csv(file).itertuples():
            known_phrases.append({'text': getattr(r, col), 'ro': r.ro, 'lang': lang, 'source': r.source,
                                  'note': getattr(r, 'heard_in', '') if file == 'romani_songs_test.csv' else ''})

web = {
    'thresholds': {'green': GREEN, 'yellow': YELLOW, 'similar': 0.75},
    'packed': pack_gloss(GLOSS),
    'phrases': known_phrases,
    'traps': {a: list(v) for a, v in TRAP.items()},
    'ro_vocab': RO_VOCAB,
    'examples': examples,
    'results': final.reset_index().to_dict(orient='records'),
    'calibration': calib.reset_index().to_dict(orient='records'),
    'context': {'chrf': chrf_rows, 'words': word_rows} if MT_OK else None,
    'counts': {a: aro_counts.get(a, 0) for a in GLOSS},   # how often each word appeared in training
    'phrasebook': pd.DataFrame({name: {p: evaluate(part, f, aro_counts)['lenient'] for p, part in parts.items()}
                                for name, f in methods.items()}).T.reset_index().to_dict(orient='records') if os.path.exists('phrasebook_test.csv') else None,
}
# Lav: the Romani model, the language detector and the mix-up list go in a second file,
# so the page can show Aromanian cards before the Romani part has loaded.
web_rom = {
    'rom': {'packed': pack_gloss(R_GLOSS),
            'counts': dict(rom_counts),
            'ro_vocab': R_RO_VOCAB,
            'thresholds': {'green': R_GREEN, 'yellow': R_YELLOW, 'similar': 0.75},
            'calibration': r_calib.reset_index().to_dict(orient='records'),
            'results': rom_final.reset_index().to_dict(orient='records'),
            'songs': rsong[['rom', 'ro', 'heard_in']].to_dict(orient='records') if 'rsong' in globals() else [],
            'examples': [{'label': 'Everyday', 'rom': 'Socares, prala? Mishto san?'},
                         {'label': 'From songs', 'rom': 'Tu san shukar, chaiorie! Kamav tut.'},
                         {'label': 'Bible verse', 'rom': r_test.rom.iloc[0]}]},
    'langid': LID,
    'langid_results': {'table': lid_table.reset_index().to_dict(orient='records'), 'everyday': lid_everyday},
    'mixups': mixups.head(30).to_dict(orient='records'),
}
# Saved as small JavaScript files (window.ZBOR_ARO = {...}), not plain .json: a browser lets a page opened straight
# from a folder (file://) load scripts next to it, but not .json files. So index.html works with a double-click.
for name, var, obj in [('zbor_aro.js', 'ZBOR_ARO', web), ('zbor_rom.js', 'ZBOR_ROM', web_rom)]:
    with open(name, 'w', encoding='utf-8') as f:
        f.write(f'window.{var} = ' + json.dumps(obj, ensure_ascii=False, separators=(',', ':')) + ';\n')
    print(f"{name}: {os.path.getsize(name) / 1e6:.2f} MB")
print('Copy zbor_aro.js and zbor_rom.js next to website/index.html.')
print(f"{len(GLOSS):,} Aromanian words | {len(R_GLOSS):,} Romani words | {len(web['traps'])} traps | {len(known_phrases)} sourced phrases")
print([e['label'] for e in examples])

zbor_aro.js: 3.30 MB
zbor_rom.js: 1.71 MB
Copy zbor_aro.js and zbor_rom.js next to website/index.html.
48,632 Aromanian words | 15,382 Romani words | 177 traps | 105 sourced phrases
['zbor in a sentence', 'Folk tale 1', 'Folk tale 2', 'Folk tale 3', 'Aromanian Wikipedia']


## 19. Limitations and next steps

- **No native-speaker check yet.** The traps and glosses still need to be confirmed by Aromanian speakers.
- **No context.** Each word gets one meaning, so words with two meanings get only the most common one.
- **Old, religious training data.** Modern words are often unknown. Adding modern Aromanian text (with permission) would help most.
- **Automatic scoring is approximate.** "Matches a word in the reference translation" is not the same as "is the right translation".
- **Related work.** AroTranslate (Jerpelea et al., 2025) already translates whole sentences. Zbor is different: it explains word by word and flags traps, for learners who want to understand the text, not just get a translation.
- **Lav is rarely "sure".** Only about 1 Romani word in 10 reaches the green level (right 80% of the time on test); most show "check" or "guess". Its training text is two Bibles, about 4 times less than Zbor's, and slang like *Socares* still comes out wrong.
- **The data check found problems but removing them barely matters.** Dropping the 579 flagged pairs changed dev by at most 0.4 points (dictionary words 22.8% → 23.2%, about two words), so we kept all pairs. Fixing the clear errors by hand is a next step.
- **Google Translate** has no Aromanian. It added Romani in 2024, but it translates whole sentences and does not explain words or tell the two languages apart.

## Sources
- Main corpus: `aronlp/aromanian-romanian-MT-corpus-limited`, CC BY-NC 4.0. Petrariu & Nisioi, *A Multilingual Parallel Corpus for Aromanian*, LREC-COLING 2024; Jerpelea, Rădoi & Nisioi, *Dialectal and Low Resource Machine Translation for Aromanian*, COLING 2025.
- Folk tales: github.com/senisioi/aromanian (GPL-3.0), same authors.
- IBM Model 1: Brown et al., *The Mathematics of Statistical Machine Translation*, 1993.
- Aromanian Wikipedia (roa-rup.wikipedia.org), CC BY-SA 4.0.
- Romani Bible (Kalderash): OPUS bible-uedin ro–rom, CC0. Christodoulopoulos & Steedman, *A massively parallel corpus: the Bible in 100 languages*, 2015.
- Chergash Romani New Testament, © 2018 Pioneers, ebible.org/details.php?id=rmyChergash, CC BY-SA 4.0; Biblia Liberă (Romanian), ebible.org/details.php?id=ronbl, public domain. We did not change the texts, only paired them by verse.
- Everyday and song test sets: sources listed line by line in the three test files at the top of this notebook.
- Libraries: pandas, NumPy, rapidfuzz, sacrebleu, CTranslate2, Hugging Face datasets, Gradio.
- "zbor" = "word" in Aromanian: Wiktionary.
- AI tools: Claude (helped us choose the project, research the data, draft and debug the notebook code, and draft the write-up) and ChatGPT _[say what you used it for]_. We ran every cell ourselves, checked the results and can explain each part.